# Parte 1 · Nem todo exemplo é igualmente difícil

**Avaliação com habilidade latente · Instituto Kunumi**

Três classificadores com quase a mesma acurácia erram justamente nos mesmos poucos exemplos. A tabela de acertos e erros que aparece aqui é a matéria-prima da IRT.

**Execute a célula de preparação primeiro.** Depois siga as tarefas em ordem: em cada uma, mude *uma* coisa, observe o resultado e responda *uma* pergunta.

In [ ]:
import subprocess
import sys
from pathlib import Path

# No Colab, baixa o repositório do tutorial; localmente, encontra a raiz a partir desta pasta.
if "google.colab" in sys.modules:
    ROOT = Path("/content/latent-ability-ml")
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/manuelfjr/latent-ability-ml.git", str(ROOT)])
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "handson.py").exists())

sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "figure.facecolor": "#F0F0F0",
    "axes.facecolor": "#F0F0F0",
    "savefig.facecolor": "#F0F0F0",
    "axes.edgecolor": "#B4ADA4",
    "axes.labelcolor": "#1C2127",
    "axes.titlecolor": "#1C2127",
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#B4ADA4",
    "grid.alpha": 0.35,
    "xtick.color": "#5E5E5E",
    "ytick.color": "#5E5E5E",
    "text.color": "#1C2127",
    "legend.frameon": False,
    "lines.linewidth": 2.6,
    "axes.prop_cycle": mpl.cycler(color=["#F04E44", "#4392D2", "#2FB7CF", "#62CCA3", "#F0D449"]),
})

## Tarefa 1 · Olhe os dados (~2 min)

Execute a célula. Ela gera um problema de classificação com duas classes.

**Pergunta.** Onde as duas classes se misturam? Quais pontos você acha que um modelo vai errar?

In [ ]:
from utils.handson import make_toy_classification_dataset

df = make_toy_classification_dataset(n_samples=300, class_sep=1.4, random_state=56)

fig, ax = plt.subplots()
for label, grupo in df.groupby("label"):
    ax.scatter(grupo.feature_1, grupo.feature_2, s=18, alpha=0.8, label=f"classe {label}")
ax.set_title("Duas classes")
ax.legend()
plt.show()

*Sua resposta:* 

## Tarefa 2 · Treine três modelos (~3 min)

Execute. A célula treina uma árvore de decisão, um k-vizinhos e uma regressão logística e mostra a acurácia de cada um no teste.

**Pergunta.** Qual modelo ganhou? A diferença é grande o suficiente para importar?

In [ ]:
from utils.handson import evaluate_models_on_dataset, summarize_classification_results

resultados = evaluate_models_on_dataset(df, scenario="ita", random_state=56)

summarize_classification_results(resultados).set_index("model")[["accuracy"]].round(3)

*Sua resposta:* 

## Tarefa 3 · Quem acertou o quê? (~3 min)

Execute. Cada linha é um modelo e cada coluna é um exemplo de teste: `1` é acerto, `0` é erro. Aparecem só os 12 primeiros exemplos.

**Pergunta.** Encontre um exemplo que todos acertam e um exemplo em que os modelos discordam.

In [ ]:
respostas = resultados.pivot(index="model", columns="example_id", values="correct")

print(respostas.shape, "→ (modelos, exemplos de teste)")
respostas.iloc[:, :12]

*Sua resposta:* 

## Tarefa 4 · Meça a dificuldade (~3 min)

A dificuldade de um exemplo pode ser medida como a fração de modelos que erraram. Execute para listar os mais difíceis.

**Pergunta.** Quantos exemplos todos os modelos erram? E quantos ninguém erra?

In [ ]:
dificuldade = 1 - respostas.mean(axis=0)   # fração de modelos que erraram

print("errados por todos:  ", int((dificuldade == 1).sum()))
print("errados por ninguém:", int((dificuldade == 0).sum()))
dificuldade.sort_values(ascending=False).head(8).round(2)

*Sua resposta:* 

## Tarefa 5 · Veja onde eles estão (~3 min)

Execute para destacar, no mapa, os exemplos que pelo menos um modelo errou.

**Pergunta.** Os exemplos difíceis estão espalhados ao acaso ou concentrados em alguma região?

In [ ]:
dificeis = dificuldade[dificuldade > 0].index
teste = df[df.example_id.isin(respostas.columns)]
facil = teste[~teste.example_id.isin(dificeis)]
dificil = teste[teste.example_id.isin(dificeis)]

fig, ax = plt.subplots()
for label, cor in [(0, "#B4ADA4"), (1, "#5E5E5E")]:
    grupo = facil[facil.label == label]
    ax.scatter(grupo.feature_1, grupo.feature_2, s=16, color=cor, label=f"classe {label}: todos acertam")
ax.scatter(dificil.feature_1, dificil.feature_2, s=90, color="#F04E44", label="algum modelo erra")
ax.set_title("Onde os modelos erram")
ax.legend()
plt.show()

*Sua resposta:* 

## Desafio · Desafio: deixe o problema mais difícil (~3 min)

Esta célula refaz tudo de uma vez. Mude `class_sep` de `1.4` para `0.6` e execute.

**Pergunta.** O que acontece com o número de exemplos difíceis e com a diferença entre os modelos?

In [ ]:
class_sep = 1.4   # <- mude aqui

dados = make_toy_classification_dataset(n_samples=300, class_sep=class_sep, random_state=56)
res = evaluate_models_on_dataset(dados, scenario="desafio", random_state=56)
tabela = res.pivot(index="model", columns="example_id", values="correct")

print("exemplos difíceis:", int((tabela.mean(axis=0) < 1).sum()), "de", tabela.shape[1])
tabela.mean(axis=1).round(3).rename("acurácia")

*Sua resposta:* 